# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows.
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [3]:
#1
class matrix:
    def __init__(self, n, m=None):
        if m is not None:
            if type(n) is not int or type(m) is not int:
                raise TypeError("Dimensions must be integers.")

            if n < 0 or m < 0:
                raise ValueError("Dimensions cannot be negative.")

            self.n = n
            self.m = m
            self.data = [[0 for j in range(m)] for i in range(n)]

        else:
            if not isinstance(n, list):
                raise TypeError("Provide a list of lists.")

            if any(not isinstance(row, list) for row in n):
                raise TypeError("Every row must be a list.")

            columns = len(n[0]) if n else 0

            if any(len(row) != columns for row in n):
                raise ValueError("All rows must have equal length.")

            self.n = len(n)
            self.m = columns
            self.data = [row[:] for row in n]

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value

        elif isinstance(key, slice):
            if key != slice(None):
                raise ValueError("Use [:] for whole-matrix assignment.")
            self.assign(value)

        else:
            if not isinstance(value, list):
                raise TypeError("A row must be a list.")

            if len(value) != self.m:
                raise ValueError("Incorrect row length.")

            self.data[key] = value[:]

    def assign(self, other):
        if isinstance(other, matrix):
            rows, columns = other.n, other.m
            values = [row[:] for row in other.data]
        else:
            other = matrix(other)
            rows, columns = other.n, other.m
            values = [row[:] for row in other.data]

        if (self.n, self.m) != (rows, columns):
            raise ValueError("Matrices must have the same size.")

        self.data = values

    def __repr__(self):
        return f"matrix({self.data})"

In [4]:
def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        print(f"Expected {error_type.__name__} caught.")
    else:
        raise AssertionError(f"Expected {error_type.__name__}.")


A = matrix(2, 3)
assert A.data == [[0, 0, 0], [0, 0, 0]]

# Rows must be independent
A[0][0] = 9
assert A[1][0] == 0

# Initialization with lists
source = [[1, 2, 3], [4, 5, 6]]
B = matrix(source)
source[0][0] = 99
assert B[0][0] == 1

# Both indexing formats
assert B[1][2] == 6
assert B[1, 2] == 6

# Both element formats
A[0][1] = 8
A[1, 2] = 7
assert A[0, 1] == 8
assert A[1][2] == 7

# Assignment from a matrix
A.assign(B)
assert A.data == B.data

A[0, 0] = 100
assert B[0, 0] == 1

# Assigning from lists
A.assign([[7, 8, 9], [10, 11, 12]])
assert A[1, 2] == 12

# Whole matrix using [:]
A[:] = B
assert A.data == B.data

A[:] = [[2, 2, 2], [3, 3, 3]]
assert A.data == [[2, 2, 2], [3, 3, 3]]

# Invalid inputs and incompatible sizes
expect_error(ValueError, lambda: matrix([[1, 2], [3]]))
expect_error(TypeError, lambda: matrix([1, 2]))
expect_error(ValueError, lambda: matrix(-1, 2))
expect_error(TypeError, lambda: matrix(2.5, 3))
expect_error(ValueError, lambda: A.assign(matrix(3, 2)))
expect_error(ValueError, lambda: A.assign([[1]]))

print("Question 1: All tests passed!")

Expected ValueError caught.
Expected TypeError caught.
Expected ValueError caught.
Expected TypeError caught.
Expected ValueError caught.
Expected ValueError caught.
Question 1: All tests passed!


In [5]:
#2
class matrix(matrix):
    def shape(self):
        return (self.n, self.m)

    def to_list(self):
        return [row[:] for row in self.data]

    def transpose(self):
        result = matrix(self.m, self.n)

        for i in range(self.n):
            for j in range(self.m):
                result[j, i] = self.data[i][j]

        return result

    def _indices(self, key, size):
        if isinstance(key, slice):
            return list(range(size))[key]

        if isinstance(key, int):
            if key < 0:
                key += size

            if key < 0 or key >= size:
                raise IndexError("Matrix index out of range.")

            return [key]

        raise TypeError("Indices must be integers or slices.")

    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("Use two indices.")

            row_key, column_key = key
            rows = self._indices(row_key, self.n)
            columns = self._indices(column_key, self.m)

            if isinstance(row_key, int) and isinstance(column_key, int):
                return self.data[rows[0]][columns[0]]

            result = matrix(len(rows), len(columns))
            result.data = [
                [self.data[i][j] for j in columns]
                for i in rows
            ]
            return result

        if isinstance(key, slice):
            return self[key, :]

        i = self._indices(key, self.n)[0]
        return self.data[i]

    def row(self, n):
        return self[n, :]

    def column(self, n):
        return self[:, n]

    def block(self, n_0, n_1, m_0, m_1):
        return self[m_0:m_1, n_0:n_1]

In [6]:
A = matrix([
    [1, 2, 3],
    [4, 5, 6]
])

assert A.shape() == (2, 3)
assert A.to_list() == [[1, 2, 3], [4, 5, 6]]

T = A.transpose()
assert isinstance(T, matrix)
assert T.shape() == (3, 2)
assert T.to_list() == [[1, 4], [2, 5], [3, 6]]

R = A.row(1)
assert isinstance(R, matrix)
assert R.shape() == (1, 3)
assert R.to_list() == [[4, 5, 6]]

C = A.column(1)
assert isinstance(C, matrix)
assert C.shape() == (2, 1)
assert C.to_list() == [[2], [5]]

# Columns 1:3 and rows 0:2
assert A.block(1, 3, 0, 2).to_list() == [[2, 3], [5, 6]]

# Slicing
assert A[:1].to_list() == [[1, 2, 3]]
assert A[:, 1:].to_list() == [[2, 3], [5, 6]]
assert A[0, :].to_list() == [[1, 2, 3]]
assert A[:, 0].to_list() == [[1], [4]]
assert A[:, ::2].to_list() == [[1, 3], [4, 6]]
assert A[::-1, ::-1].to_list() == [[6, 5, 4], [3, 2, 1]]

# Original indexing works
assert A[1][2] == 6
assert A[1, 2] == 6
assert A[-1, -1] == 6

# Returned objects are independent copies
T[0, 0] = 99
R[0, 0] = 99
C[0, 0] = 99
values = A.to_list()
values[0][0] = 99
assert A.to_list() == [[1, 2, 3], [4, 5, 6]]

# Empty shapes and invalid indices
assert A[:0, :].shape() == (0, 3)
assert A[:, :0].shape() == (2, 0)
assert matrix(0, 3).transpose().shape() == (3, 0)

expect_error(IndexError, lambda: A[2, 0])
expect_error(IndexError, lambda: A[0, 3])

print("Question 2: All tests passed!")

Expected IndexError caught.
Expected IndexError caught.
Question 2: All tests passed!


In [7]:
#3
def constant(n, m, c):
    result = matrix(n, m)
    value = float(c)

    for i in range(n):
        for j in range(m):
            result[i, j] = value

    return result


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    result = zeros(n, n)

    for i in range(n):
        result[i, i] = 1.0

    return result

In [8]:
K = constant(2, 3, 5)
Z = zeros(2, 3)
O = ones(2, 3)
I = eye(3)

assert K.to_list() == [[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]]
assert Z.to_list() == [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
assert O.to_list() == [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]

assert I.to_list() == [
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
]

assert K.shape() == (2, 3)
assert Z.shape() == (2, 3)
assert O.shape() == (2, 3)
assert I.shape() == (3, 3)

for M in (K, Z, O, I):
    assert isinstance(M, matrix)
    assert all(
        isinstance(value, float)
        for row in M.to_list()
        for value in row
    )

expect_error(ValueError, lambda: zeros(-1, 2))
expect_error(TypeError, lambda: ones(2.5, 3))

print("Question 3: All tests passed!")

Expected ValueError caught.
Expected TypeError caught.
Question 3: All tests passed!


In [9]:
#4
from numbers import Number


class matrix(matrix):
    def _check_same_size(self, other):
        if not isinstance(other, matrix):
            raise TypeError("The other object must be a matrix.")

        if self.shape() != other.shape():
            raise ValueError("Matrices must have the same dimensions.")

    def scalarmul(self, c):
        if not isinstance(c, Number):
            raise TypeError("The scalar must be a number.")

        result = matrix(self.n, self.m)

        for i in range(self.n):
            for j in range(self.m):
                result[i, j] = self[i, j] * c

        return result

    def add(self, other):
        self._check_same_size(other)
        result = matrix(self.n, self.m)

        for i in range(self.n):
            for j in range(self.m):
                result[i, j] = self[i, j] + other[i, j]

        return result

    def sub(self, other):
        self._check_same_size(other)
        result = matrix(self.n, self.m)

        for i in range(self.n):
            for j in range(self.m):
                result[i, j] = self[i, j] - other[i, j]

        return result

    def mat_mult(self, other):
        if not isinstance(other, matrix):
            raise TypeError("The other object must be a matrix.")

        if self.m != other.n:
            raise ValueError(
                "Columns of the first matrix must equal "
                "rows of the second matrix."
            )

        result = matrix(self.n, other.m)

        for i in range(self.n):
            for j in range(other.m):
                for k in range(self.m):
                    result[i, j] += self[i, k] * other[k, j]

        return result

    def element_mult(self, other):
        self._check_same_size(other)
        result = matrix(self.n, self.m)

        for i in range(self.n):
            for j in range(self.m):
                result[i, j] = self[i, j] * other[i, j]

        return result

    def equals(self, other):
        if not isinstance(other, matrix):
            return False

        return (
            self.shape() == other.shape()
            and self.data == other.data
        )

In [10]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

assert A.scalarmul(2).to_list() == [[2, 4], [6, 8]]
assert A.add(B).to_list() == [[6, 8], [10, 12]]
assert A.sub(B).to_list() == [[-4, -4], [-4, -4]]
assert A.mat_mult(B).to_list() == [[19, 22], [43, 50]]
assert A.element_mult(B).to_list() == [[5, 12], [21, 32]]

assert A.equals(matrix([[1, 2], [3, 4]]))
assert not A.equals(B)
assert not A.equals(matrix(3, 2))
assert not A.equals([[1, 2], [3, 4]])

# Rectangular matrix multiplication
R = matrix([[1, 2, 3], [4, 5, 6]])
S = matrix([[7, 8], [9, 10], [11, 12]])

product = R.mat_mult(S)
assert product.shape() == (2, 2)
assert product.to_list() == [[58, 64], [139, 154]]

# Incompatible dimensions
wrong = matrix(3, 2)

expect_error(ValueError, lambda: A.add(wrong))
expect_error(ValueError, lambda: A.sub(wrong))
expect_error(ValueError, lambda: A.element_mult(wrong))
expect_error(ValueError, lambda: A.mat_mult(wrong))
expect_error(TypeError, lambda: A.scalarmul("hello"))

# Original matrices must remain unchanged
assert A.to_list() == [[1, 2], [3, 4]]
assert B.to_list() == [[5, 6], [7, 8]]

print("Question 4: All tests passed!")

Expected ValueError caught.
Expected ValueError caught.
Expected ValueError caught.
Expected ValueError caught.
Expected TypeError caught.
Question 4: All tests passed!


In [11]:
#5
class matrix(matrix):
    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)

        if isinstance(other, Number):
            return self.scalarmul(other)

        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, Number):
            return self.scalarmul(other)

        return NotImplemented

    def __eq__(self, other):
        return self.equals(other)

In [12]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

assert (2 * A).to_list() == [[2, 4], [6, 8]]
assert (A * 2).to_list() == [[2, 4], [6, 8]]
assert (A + B).to_list() == [[6, 8], [10, 12]]
assert (A - B).to_list() == [[-4, -4], [-4, -4]]
assert (A * B).to_list() == [[19, 22], [43, 50]]

assert A == matrix([[1, 2], [3, 4]])
assert not (A == B)
assert A != B
assert not (A == matrix(3, 2))

wrong = matrix(3, 2)

expect_error(ValueError, lambda: A + wrong)
expect_error(ValueError, lambda: A - wrong)
expect_error(ValueError, lambda: A * wrong)
expect_error(TypeError, lambda: A * "hello")

print("Question 5: All tests passed!")

Expected ValueError caught.
Expected ValueError caught.
Expected ValueError caught.
Expected TypeError caught.
Question 5: All tests passed!


In [13]:
#6

A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])
C = matrix([[2, 0], [1, 2]])
I = eye(2)

# Associative property
associative_left = (A * B) * C
associative_right = A * (B * C)

print("Associative property:")
print("(AB)C =", associative_left)
print("A(BC) =", associative_right)

# Distributive property
distributive_left = A * (B + C)
distributive_right = A * B + A * C

print("\nDistributive property:")
print("A(B + C) =", distributive_left)
print("AB + AC  =", distributive_right)

# Multiplication is not commutative in general
AB = A * B
BA = B * A

print("\nNoncommutative example:")
print("AB =", AB)
print("BA =", BA)

# Identity property
AI = A * I

print("\nIdentity property:")
print("AI =", AI)
print("A  =", A)

Associative property:
(AB)C = matrix([[60, 44], [136, 100]])
A(BC) = matrix([[60, 44], [136, 100]])

Distributive property:
A(B + C) = matrix([[23, 26], [53, 58]])
AB + AC  = matrix([[23, 26], [53, 58]])

Noncommutative example:
AB = matrix([[19, 22], [43, 50]])
BA = matrix([[23, 34], [31, 46]])

Identity property:
AI = matrix([[1.0, 2.0], [3.0, 4.0]])
A  = matrix([[1, 2], [3, 4]])


In [14]:
assert associative_left == associative_right
assert associative_left.to_list() == [[60, 44], [136, 100]]
print("(AB)C = A(BC): Passed")

assert distributive_left == distributive_right
assert distributive_left.to_list() == [[23, 26], [53, 58]]
print("A(B + C) = AB + AC: Passed")

assert AB != BA
assert AB.to_list() == [[19, 22], [43, 50]]
assert BA.to_list() == [[23, 34], [31, 46]]
print("AB != BA for these matrices: Passed")

assert AI == A
print("AI = A: Passed")

print("\nQuestion 6: All tests passed!")

(AB)C = A(BC): Passed
A(B + C) = AB + AC: Passed
AB != BA for these matrices: Passed
AI = A: Passed

Question 6: All tests passed!
